In [1]:
# ============================================================
# E6 — Explanation-only SFT
# ============================================================

!pip -q install -U transformers peft accelerate bitsandbytes sentencepiece

import os, json, re, torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model
from google.colab import drive

drive.mount("/content/drive")
# ----------------------------
# CONFIG
# ----------------------------
BASE_DIR = "/content/drive/MyDrive/DL_Local"
PROMPTS_PATH = os.path.join(BASE_DIR, "clinical_pharm_prompts_10000.jsonl")
TEACHER_PATH = os.path.join(BASE_DIR, "clinical_pharm_teacher_train_6000.jsonl")
OUT_DIR = os.path.join(BASE_DIR, "e6_explanation_only_sft")
os.makedirs(OUT_DIR, exist_ok=True)

MAX_SEQ_LEN = 2048
EPOCHS = 2
LR = 2e-4
BATCH = 1
GRAD_ACC = 32

STUDENTS = {
    "qwen25_1p5b_base":     {"path": "Qwen/Qwen2.5-1.5B",          "is_instruct": False},
    "qwen25_1p5b_instruct": {"path": "Qwen/Qwen2.5-1.5B-Instruct", "is_instruct": True},
    "qwen25_3b_base":       {"path": "Qwen/Qwen2.5-3B",            "is_instruct": False},
    "qwen25_3b_instruct":   {"path": "Qwen/Qwen2.5-3B-Instruct",   "is_instruct": True},
    "qwen25_7b_base":       {"path": "Qwen/Qwen2.5-7B",            "is_instruct": False},
    "qwen25_7b_instruct":   {"path": "Qwen/Qwen2.5-7B-Instruct",   "is_instruct": True},
}


# ----------------------------
# Load data
# ----------------------------
def load_jsonl(p):
    with open(p, "r", encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

prompts = {r["id"]: r["prompt"] for r in load_jsonl(PROMPTS_PATH)}
rows = [r for r in load_jsonl(TEACHER_PATH) if r["status"] == "ok"]

# ----------------------------
# Explanation span
# ----------------------------
EXP_PAT = [r"\bExplanation\s*:\s*", r'"explanation"\s*:\s*']

def find_expl_span(txt):
    for p in EXP_PAT:
        m = re.search(p, txt)
        if m:
            return (m.end(), len(txt))
    return None

# ----------------------------
# Dataset
# ----------------------------
class E6Dataset(Dataset):
    def __init__(self, rows, tok, is_instr):
        self.rows, self.tok, self.is_instr = rows, tok, is_instr

    def __len__(self): return len(self.rows)

    def __getitem__(self, idx):
        r = self.rows[idx]
        prompt = prompts[r["id"]]
        ans = r["teacher_text"]

        if self.is_instr and hasattr(self.tok, "apply_chat_template"):
            ptxt = self.tok.apply_chat_template(
                [{"role": "user", "content": prompt}],
                tokenize=False,
                add_generation_prompt=True
            )
        else:
            ptxt = prompt

        full = ptxt + ans
        enc = self.tok(full, truncation=True, max_length=MAX_SEQ_LEN, return_offsets_mapping=True)
        ids, offs = enc["input_ids"], enc["offset_mapping"]

        labels = [-100] * len(ids)
        expl = find_expl_span(ans)
        start = len(ptxt)

        if expl:
            es, ee = expl
            for i, (s, _) in enumerate(offs):
                if start + es <= s < start + ee:
                    labels[i] = ids[i]

        return {
            "input_ids": torch.tensor(ids),
            "attention_mask": torch.ones(len(ids)),
            "labels": torch.tensor(labels),
        }

# ----------------------------
# Train
# ----------------------------
for name, cfg in STUDENTS.items():
    tok = AutoTokenizer.from_pretrained(cfg["path"], trust_remote_code=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        cfg["path"], load_in_4bit=True, device_map="auto", trust_remote_code=True
    )
    model = get_peft_model(model, LoraConfig(
        r=16, lora_alpha=32,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        lora_dropout=0.05,
        task_type="CAUSAL_LM"
    ))

    trainer = Trainer(
        model=model,
        args=TrainingArguments(
            output_dir=os.path.join(OUT_DIR, name),
            num_train_epochs=EPOCHS,
            per_device_train_batch_size=BATCH,
            gradient_accumulation_steps=GRAD_ACC,
            learning_rate=LR,
            fp16=True,
            remove_unused_columns=False,
            report_to="none",
        ),
        train_dataset=E6Dataset(rows, tok, cfg["is_instruct"]),
    )

    trainer.train()
    model.save_pretrained(os.path.join(OUT_DIR, name))


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 139.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 46.7 MB/s eta 0:00:00
Mounted at /content/drive


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model-00001-of-00004.safetensors:   0%|          | 0.00/3.95G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Step,Training Loss


In [1]:
# ============================================================
# E7 — Decision-only SFT (SELF-CONTAINED + RESUME-SAFE)
# ✅ SAME TRAINING LOGIC as your E7 (same dataset/labels objective)
# ✅ Adds only: checkpoint saving + auto-resume + "completed" marker
# If no checkpoint exists (because old run didn't save), it starts from scratch.
# ============================================================

!pip -q install -U transformers peft accelerate bitsandbytes sentencepiece

import os, json, re, torch, glob
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model
from google.colab import drive

drive.mount("/content/drive")

# ----------------------------
# CONFIG (unchanged)
# ----------------------------
BASE_DIR = "/content/drive/MyDrive/DL_Local"
PROMPTS_PATH = os.path.join(BASE_DIR, "clinical_pharm_prompts_10000.jsonl")
TEACHER_PATH = os.path.join(BASE_DIR, "clinical_pharm_teacher_train_6000.jsonl")
OUT_DIR = os.path.join(BASE_DIR, "e7_decision_only_sft")
os.makedirs(OUT_DIR, exist_ok=True)

MAX_SEQ_LEN = 2048
EPOCHS = 2
LR = 2e-4
BATCH = 1
GRAD_ACC = 32

STUDENTS = {
    "qwen25_1p5b_base":     {"path": "Qwen/Qwen2.5-1.5B",          "is_instruct": False},
    "qwen25_1p5b_instruct": {"path": "Qwen/Qwen2.5-1.5B-Instruct", "is_instruct": True},
    "qwen25_3b_base":       {"path": "Qwen/Qwen2.5-3B",            "is_instruct": False},
    "qwen25_3b_instruct":   {"path": "Qwen/Qwen2.5-3B-Instruct",   "is_instruct": True},
    "qwen25_7b_base":       {"path": "Qwen/Qwen2.5-7B",            "is_instruct": False},
    "qwen25_7b_instruct":   {"path": "Qwen/Qwen2.5-7B-Instruct",   "is_instruct": True},
}

# ----------------------------
# Load data (unchanged)
# ----------------------------
def load_jsonl(p):
    with open(p, "r", encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

prompts = {r["id"]: r["prompt"] for r in load_jsonl(PROMPTS_PATH)}
rows = [r for r in load_jsonl(TEACHER_PATH) if r["status"] == "ok"]

# ----------------------------
# Decision span (unchanged)
# ----------------------------
DEC_PAT = [r"\bDecision\s*:\s*", r'"decision"\s*:\s*']

def find_dec_span(txt):
    for p in DEC_PAT:
        m = re.search(p, txt)
        if m:
            return (m.end(), len(txt))
    return None

# ----------------------------
# Dataset (unchanged)
# ----------------------------
class E7Dataset(Dataset):
    def __init__(self, rows, tok, is_instr):
        self.rows, self.tok, self.is_instr = rows, tok, is_instr

    def __len__(self): return len(self.rows)

    def __getitem__(self, idx):
        r = self.rows[idx]
        prompt = prompts[r["id"]]
        ans = r["teacher_text"]

        if self.is_instr and hasattr(self.tok, "apply_chat_template"):
            ptxt = self.tok.apply_chat_template(
                [{"role": "user", "content": prompt}],
                tokenize=False,
                add_generation_prompt=True
            )
        else:
            ptxt = prompt

        full = ptxt + ans
        enc = self.tok(full, truncation=True, max_length=MAX_SEQ_LEN, return_offsets_mapping=True)
        ids, offs = enc["input_ids"], enc["offset_mapping"]

        labels = [-100] * len(ids)
        dec = find_dec_span(ans)
        start = len(ptxt)

        if dec:
            ds, de = dec
            for i, (s, _) in enumerate(offs):
                if start + ds <= s < start + de:
                    labels[i] = ids[i]

        return {
            "input_ids": torch.tensor(ids),
            "attention_mask": torch.ones(len(ids)),
            "labels": torch.tensor(labels),
        }

# ----------------------------
# Resume helpers (NEW, no logic change)
# ----------------------------
SAVE_STEPS = 100          # checkpoint frequency (safe; doesn't change training objective)
SAVE_TOTAL_LIMIT = 3      # keep last few checkpoints

def latest_checkpoint(run_dir: str):
    ckpts = glob.glob(os.path.join(run_dir, "checkpoint-*"))
    if not ckpts:
        return None
    def step(p):
        m = re.search(r"checkpoint-(\d+)$", p)
        return int(m.group(1)) if m else -1
    ckpts = sorted(ckpts, key=step)
    return ckpts[-1]

def completed_marker(run_dir: str):
    return os.path.join(run_dir, "completed.ok")

# ----------------------------
# Train (same loop; only adds resume + checkpointing)
# ----------------------------
for name, cfg in STUDENTS.items():
    run_dir = os.path.join(OUT_DIR, name)
    os.makedirs(run_dir, exist_ok=True)

    # Skip if already finished in a previous run
    if os.path.exists(completed_marker(run_dir)):
        print(f"✅ {name}: already completed, skipping.")
        continue

    tok = AutoTokenizer.from_pretrained(cfg["path"], trust_remote_code=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        cfg["path"], load_in_4bit=True, device_map="auto", trust_remote_code=True
    )
    model = get_peft_model(model, LoraConfig(
        r=16, lora_alpha=32,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        lora_dropout=0.05,
        task_type="CAUSAL_LM"
    ))

    args = TrainingArguments(
        output_dir=run_dir,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH,
        gradient_accumulation_steps=GRAD_ACC,
        learning_rate=LR,
        fp16=True,
        remove_unused_columns=False,
        report_to="none",

        # NEW: checkpointing for resume
        save_strategy="steps",
        save_steps=SAVE_STEPS,
        save_total_limit=SAVE_TOTAL_LIMIT,
        logging_steps=25,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=E7Dataset(rows, tok, cfg["is_instruct"]),
    )

    ckpt = latest_checkpoint(run_dir)
    if ckpt:
        print(f"🔄 {name}: resuming from {ckpt}")
        trainer.train(resume_from_checkpoint=ckpt)
    else:
        print(f"▶️ {name}: starting from scratch (no checkpoint found)")
        trainer.train()

    # Save final adapter weights (same as your code, just to run_dir)
    model.save_pretrained(run_dir)

    # Mark completion so reruns don't retrain finished students
    with open(completed_marker(run_dir), "w") as f:
        f.write("ok\n")

    print(f"✅ {name}: finished and saved to {run_dir}")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 154.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 45.1 MB/s eta 0:00:00
Mounted at /content/drive


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

▶️ qwen25_7b_instruct: starting from scratch (no checkpoint found)


Step,Training Loss
25,1.234500
50,1.022600
75,0.959800
100,0.910200
125,0.883400
150,0.851200
175,0.829700
200,0.792900
225,0.781000
250,0.785300


✅ qwen25_7b_instruct: finished and saved to /content/drive/MyDrive/DL_Local/e7_decision_only_sft/qwen25_7b_instruct


 ▶️ qwen25_7b_instruct: starting from scratch (no checkpoint found)
[ 51/314 24:52 < 2:13:30, 0.03 it/s, Epoch 0.32/2]


In [2]:
# ============================================================
# COLAB — Unassign runtime (official API)
# ============================================================

from google.colab import runtime

print("✅ Job finished. Unassigning runtime...")
runtime.unassign()


✅ Job finished. Unassigning runtime...
